# SomniGuard — Compressed-Channel AI Model (Edge AI, EFR32xG26)
### Ordinal Severity Detection from Pre-Windowed PPG Features — Normal / Abnormal / Emergency

**Task:** classify a `(60, 28)` window into 3 **ordinal** severity classes: `0 = Normal`, `1 = Abnormal`, `2 = Emergency`
(Emergency being strictly more severe than Abnormal). The model architecture and its single output neuron are
**unchanged** from the previous binary version: `Dense(1, sigmoid)`. What changes is how that single output is
*used*: it's now treated as a continuous **severity score**, and two tuned thresholds `tau_1 < tau_2` (see
`config.json -> threshold_tuning`) turn that score into the final 3-way label at inference time. This mirrors an
ordinal classification scheme without adding output neurons, extra heads, or new layer types.

**Hardware constraint (EFR32xG26 MVP accelerator):** only `Conv1D` / `Pooling` / `Dense` layers — no LSTM/GRU/Attention-over-time. (Unchanged.)

**What changed in this version vs. the previous (binary) notebook:**
- **Labels are now 3-class ordinal** (`0/1/2`) instead of binary (`0/1`). `data.npz`'s `label_key` array is expected
  to already contain these 3 values — nothing in this notebook derives them from raw data.
- **Training objective for the single sigmoid output is now a soft ordinal regression**: instead of a binary
  `{0,1}` target, each window is trained (MSE loss) toward a fixed target value per class —
  `class_targets = [0.0, 0.7, 1.0]` for `[Normal, Abnormal, Emergency]` (`config.json -> label.training_target`).
  Spacing Abnormal's target *below* the midpoint reserves real room in `[0,1]` for Emergency, so `tau_2` has an
  actual gap to be tuned into instead of sitting against the sigmoid's saturation ceiling near 1.0 (which is what
  happened with the earlier binary-target version). The architecture itself (`Dense(1, sigmoid)`, Conv1D-only
  backbone) and the threshold-tuning module downstream are unchanged — only the training target and loss function
  (`mse` instead of `binary_crossentropy`) are different.
- **New: a three-way split.** The training portion of the data is further split into a *model-fitting* set and a
  held-out *threshold-tuning* set (`config.json -> threshold_tuning.tuning_fraction`), so `tau_1`/`tau_2` are never
  tuned on data the model was fit on, and the final test set stays untouched until final reporting.
- **New: a threshold-tuning module** (`tune_thresholds`) that grid-searches `tau_1 <= tau_2` on the tuning split to
  maximize macro-F1 across the 3 true classes (configurable metric).
- **New: 3-class metrics everywhere** — confusion matrices are now `3x3`, and precision/recall/F1 are reported
  per-class plus macro-averaged. `export.ranking_metric` changed from `f1_abnormal` to `f1_macro`.
- **Deployment artifact change:** each exported experiment's `config.json` now also carries `thresholds.tau_1` /
  `thresholds.tau_2` (in the same `[0, 1]` float score domain as the sigmoid output) alongside the existing
  normalization stats and int8 quantization parameters. The on-device C code (written by the deployment team) is
  expected to: run the compressed+quantized window through the TFLite model -> dequantize the single output using
  `output_quantization.scale`/`zero_point` -> compare the resulting float score against `tau_1`/`tau_2`.
- **`gen_compression_c.py` / the per-frame statistical compression are UNCHANGED.** Compression only touches the
  raw IR samples and has no dependency on labels or class count, so nothing there needed to change.


## Step 0 — Install & import

In [1]:
!pip install -q "tensorflow>=2.16" tf_keras pandas numpy scikit-learn


In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # MUST be set before importing tensorflow

import io
import json
import math
import shutil
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    precision_recall_fscore_support,
)

from gen_compression_c import generate_compression_c  # local helper, see gen_compression_c.py -- UNCHANGED

print(f"TensorFlow version: {tf.__version__}")


2026-09-14 18:13:10.922502: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-14 18:13:10.923387: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 18:13:11.108801: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 18:13:11.197947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-14 18:13:11.341238: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

TensorFlow version: 2.16.2


## Step 1 — Load configuration from `config.json`

In [3]:
with open("config.json") as f:
    CFG = json.load(f)

DATA_CFG = CFG["data"]
CH_CFG = CFG["channels"]
LBL_CFG = CFG["label"]
THRESH_CFG = CFG["threshold_tuning"]
COMPRESSIONS = CFG["compressions"]
MODEL_VARIANTS = CFG["model_variants"]
TRAIN_CFG = CFG["training"]
EXPORT_CFG = CFG["export"]
EXPERIMENTS = CFG["experiments"]

WINDOW_FRAMES = CH_CFG["window_frames"]        # 60
FRAME_SAMPLES = CH_CFG["frame_samples"]        # 25 raw IR sub-samples per frame
SPO2_IDX = CH_CFG["spo2_idx"]
BPM_IDX = CH_CFG["bpm_idx"]
IR_START = CH_CFG["ir_start_idx"]
IR_END = CH_CFG["ir_end_idx"]                  # exclusive
MOTION_IDX = CH_CFG["motion_idx"]
N_RAW_CHANNELS = CH_CFG["total_channels"]      # 28

NUM_CLASSES = LBL_CFG["num_classes"]           # 3
CLASS_NAMES = LBL_CFG["class_names"]           # ["Normal", "Abnormal", "Emergency"]
CLASS_VALUES = LBL_CFG["class_values"]         # [0, 1, 2]
assert NUM_CLASSES == 3 and CLASS_VALUES == [0, 1, 2], \
    "This notebook assumes the ordinal 3-class scheme 0=Normal < 1=Abnormal < 2=Emergency."

CLASS_TARGETS = np.array(LBL_CFG["training_target"]["class_targets"], dtype=np.float32)  # e.g. [0.0, 0.7, 1.0]
assert len(CLASS_TARGETS) == NUM_CLASSES, "label.training_target.class_targets must have one value per class"
assert np.all(np.diff(CLASS_TARGETS) > 0), "class_targets must be strictly increasing (it encodes the ordinal order)"

TUNING_FRACTION = THRESH_CFG["tuning_fraction"]
TUNING_SEED = THRESH_CFG["seed"]
TUNING_METRIC = THRESH_CFG["metric"]            # e.g. "f1_macro"
TUNING_N_CANDIDATES = THRESH_CFG["n_candidates"]

assert IR_END - IR_START == FRAME_SAMPLES, "ir_start/ir_end/frame_samples mismatch in config.json"

print(json.dumps(CFG, indent=2))
print(f"\nWINDOW_FRAMES={WINDOW_FRAMES} | FRAME_SAMPLES={FRAME_SAMPLES} | N_RAW_CHANNELS={N_RAW_CHANNELS}")
print(f"Raw channel layout: [spo2({SPO2_IDX}), bpm({BPM_IDX}), ir_0..ir_{FRAME_SAMPLES-1}({IR_START}:{IR_END}), motion_level({MOTION_IDX})]")
print(f"Classes: {list(zip(CLASS_VALUES, CLASS_NAMES))}")
print(f"Regression targets per class: {dict(zip(CLASS_NAMES, CLASS_TARGETS.tolist()))}")
print(f"Number of experiments configured: {len(EXPERIMENTS)}")


{
  "data": {
    "npz_path": "somniguard_train_dataset.npz",
    "data_key": "X",
    "label_key": "Y",
    "test_fraction": 0.2,
    "seed": 42
  },
  "channels": {
    "total_channels": 28,
    "spo2_idx": 0,
    "bpm_idx": 1,
    "ir_start_idx": 2,
    "ir_end_idx": 27,
    "motion_idx": 27,
    "frame_samples": 25,
    "window_frames": 60
  },
  "label": {
    "num_classes": 3,
    "class_names": [
      "Normal",
      "Abnormal",
      "Emergency"
    ],
    "class_values": [
      0,
      1,
      2
    ],
    "training_target": {
      "type": "soft_ordinal_regression",
      "class_targets": [
        0.0,
        0.7,
        1.0
      ],
      "loss": "mse",
      "description": "The Dense(1, sigmoid) output is trained with MSE regression against a fixed target value per class: class_targets[label] (0.0 for Normal, 0.7 for Abnormal, 1.0 for Emergency). Spacing Abnormal's target below the midpoint reserves more of the [0,1] range for Emergency, so tau_2 has real separation 

## Step 2 — Load the pre-windowed dataset (`data.npz`)

Expects `data.npz` next to the notebook, with `data: (N, 60, 28)` float array and `label: (N,)` int array
taking values in `{0, 1, 2}` = `{Normal, Abnormal, Emergency}`.


In [4]:
npz = np.load(DATA_CFG["npz_path"])
X_all = npz[DATA_CFG["data_key"]].astype(np.float32)
y_all = npz[DATA_CFG["label_key"]].astype(np.int64)

assert X_all.ndim == 3, f"Expected data shape (N, {WINDOW_FRAMES}, {N_RAW_CHANNELS}), got {X_all.shape}"
assert X_all.shape[1] == WINDOW_FRAMES and X_all.shape[2] == N_RAW_CHANNELS, \
    f"data.npz shape {X_all.shape} does not match config.json channels block"
assert X_all.shape[0] == y_all.shape[0], "data/label length mismatch"
assert set(np.unique(y_all).tolist()).issubset(set(CLASS_VALUES)), \
    f"y_all contains values outside {CLASS_VALUES}: {np.unique(y_all).tolist()}"

print(f"X_all: {X_all.shape} | y_all: {y_all.shape}")
labels, counts = np.unique(y_all, return_counts=True)
print(f"Label distribution: " +
      ", ".join(f"{CLASS_NAMES[l]}({l})={c}" for l, c in zip(labels.tolist(), counts.tolist())))
print(f"Sample window X_all[0][0] (first frame, all 28 channels): {X_all[0][0]}")


X_all: (4940, 60, 28) | y_all: (4940,)
Label distribution: Normal(0)=2468, Abnormal(1)=2005, Emergency(2)=467
Sample window X_all[0][0] (first frame, all 28 channels): [ 9.8000000e+01  7.5000000e+01  0.0000000e+00  3.1935604e-04
  7.4106368e-04  1.2574299e-03  1.7751971e-03  2.4142377e-03
  2.9776392e-03  3.5361655e-03  4.0874961e-03  4.6269335e-03
  5.1560337e-03  5.6720278e-03  6.1554387e-03  6.6559371e-03
  7.1167261e-03  7.5548352e-03  7.8360839e-03  8.2016690e-03
  8.5878177e-03  8.8572409e-03  8.8319015e-03  7.7464590e-03
  5.0632260e-03  1.8611058e-03 -1.1831902e-03  4.5027724e-03]


## Step 3 — Three-way split: model-fitting / threshold-tuning / test

No subject/session IDs are available in `data.npz`, so all splits below are shuffled, seeded, label-stratified
splits at the window level (stratified on the 3-class label so all three classes are represented in every split).

1. `test_fraction` (config) is held out first as the final, untouched **test** set.
2. The remainder is split again: `threshold_tuning.tuning_fraction` becomes the **tune** set used only to pick
   `tau_1`/`tau_2`; the rest is the **fit** set the model is actually trained on.

This keeps `tau_1`/`tau_2` from leaking information from the test set, and keeps the model from being fit on the
same rows used to choose the thresholds.


In [5]:
X_trainval_raw, X_test_raw, y_trainval, y_test = train_test_split(
    X_all, y_all,
    test_size=DATA_CFG["test_fraction"],
    random_state=DATA_CFG["seed"],
    stratify=y_all,
)

X_train_raw, X_tune_raw, y_train, y_tune = train_test_split(
    X_trainval_raw, y_trainval,
    test_size=TUNING_FRACTION,
    random_state=TUNING_SEED,
    stratify=y_trainval,
)


def _dist(y):
    lab, cnt = np.unique(y, return_counts=True)
    return ", ".join(f"{CLASS_NAMES[l]}({l})={c}" for l, c in zip(lab.tolist(), cnt.tolist()))


print(f"X_train_raw (fit):  {X_train_raw.shape} | dist: {_dist(y_train)}")
print(f"X_tune_raw (tune):  {X_tune_raw.shape} | dist: {_dist(y_tune)}")
print(f"X_test_raw (test):  {X_test_raw.shape} | dist: {_dist(y_test)}")


X_train_raw (fit):  (3359, 60, 28) | dist: Normal(0)=1678, Abnormal(1)=1363, Emergency(2)=318
X_tune_raw (tune):  (593, 60, 28) | dist: Normal(0)=296, Abnormal(1)=241, Emergency(2)=56
X_test_raw (test):  (988, 60, 28) | dist: Normal(0)=494, Abnormal(1)=401, Emergency(2)=93


## Step 3b — Export raw windowed fit/tune/test sets for on-device deployment testing

Saves the exact, unnormalized `(n, 60, 28)` raw windows + labels for all three splits — these are what should be
fed through the firmware's own compression + model pipeline and compared against this notebook's predictions,
independent of which compression/model variant you pick later. `tune_windows.npz` in particular is useful for the
deployment team to sanity-check that on-device dequantized scores land in the same place relative to `tau_1`/`tau_2`
as they do here.


In [6]:
np.savez_compressed("train_windows.npz", X=X_train_raw, label=y_train)
np.savez_compressed("tune_windows.npz", X=X_tune_raw, label=y_tune)
np.savez_compressed("test_windows.npz", X=X_test_raw, label=y_test)
print(f"Wrote train_windows.npz (X: {X_train_raw.shape}, label: {y_train.shape})")
print(f"Wrote tune_windows.npz  (X: {X_tune_raw.shape}, label: {y_tune.shape})")
print(f"Wrote test_windows.npz  (X: {X_test_raw.shape}, label: {y_test.shape})")
print("Channel order in X: [spo2, bpm, ir_0..ir_24, motion_level] (raw, unnormalized, uncompressed)")
print("Labels are 3-class ordinal: 0=Normal, 1=Abnormal, 2=Emergency")


Wrote train_windows.npz (X: (3359, 60, 28), label: (3359,))
Wrote tune_windows.npz  (X: (593, 60, 28), label: (593,))
Wrote test_windows.npz  (X: (988, 60, 28), label: (988,))
Channel order in X: [spo2, bpm, ir_0..ir_24, motion_level] (raw, unnormalized, uncompressed)
Labels are 3-class ordinal: 0=Normal, 1=Abnormal, 2=Emergency


## Step 4 — Statistical compression module (25 IR samples → C features, per frame)

**Unchanged from the previous version.** Collapses each frame's 25 raw `ir_*` values into a small set of
statistics, independently per timestep. Fixed formula, not learned — same math is exported to C in Step 7
(`gen_compression_c.py`, itself unmodified — it has no notion of labels/classes).

Supported feature names: `mean, std, min, max, range, rms, skew, kurtosis, mad, mean_abs_diff`. The active feature
list per compression config comes from `config.json -> compressions.<n>.features`.


In [7]:
def compress_frame_stats(X_ir, features):
    """X_ir: (..., frame_samples). Returns (..., len(features)).
    Must stay behaviourally identical to gen_compression_c.py's C output."""
    mean = X_ir.mean(axis=-1)
    std = X_ir.std(axis=-1)
    std_safe = np.where(std < 1e-8, 1.0, std)
    mn = X_ir.min(axis=-1)
    mx = X_ir.max(axis=-1)

    out = []
    for f in features:
        if f == "mean":
            out.append(mean)
        elif f == "std":
            out.append(std)
        elif f == "min":
            out.append(mn)
        elif f == "max":
            out.append(mx)
        elif f == "range":
            out.append(mx - mn)
        elif f == "rms":
            out.append(np.sqrt(np.mean(X_ir ** 2, axis=-1)))
        elif f == "skew":
            z = (X_ir - mean[..., None]) / std_safe[..., None]
            out.append(np.mean(z ** 3, axis=-1))
        elif f == "kurtosis":
            z = (X_ir - mean[..., None]) / std_safe[..., None]
            out.append(np.mean(z ** 4, axis=-1) - 3.0)
        elif f == "mad":
            out.append(np.mean(np.abs(X_ir - mean[..., None]), axis=-1))
        elif f == "mean_abs_diff":
            out.append(np.mean(np.abs(np.diff(X_ir, axis=-1)), axis=-1))
        else:
            raise ValueError(f"Unsupported feature '{f}'")
    return np.stack(out, axis=-1).astype(np.float32)


def build_compressed_tensor(X_raw, features):
    """X_raw: (N, window_frames, N_RAW_CHANNELS) -> (N, window_frames, len(features)+3)
    Output channel order: [*features, spo2, bpm, motion_level]."""
    ir = X_raw[..., IR_START:IR_END]                      # (N, window_frames, frame_samples)
    compressed = compress_frame_stats(ir, features)        # (N, window_frames, len(features))
    spo2 = X_raw[..., SPO2_IDX:SPO2_IDX + 1]
    bpm = X_raw[..., BPM_IDX:BPM_IDX + 1]
    motion = X_raw[..., MOTION_IDX:MOTION_IDX + 1]
    combined = np.concatenate([compressed, spo2, bpm, motion], axis=-1)
    return combined.astype(np.float32)


## Step 5 — Build compressed feature tensors for every compression config used by an experiment

Computed once per unique `compression` name referenced in `config.json -> experiments`, now for all **three**
splits (fit / tune / test), then reused/cached across experiments that share the same compression config.


In [8]:
compressed_cache_train = {}
compressed_cache_tune = {}
compressed_cache_test = {}

used_compressions = sorted({exp["compression"] for exp in EXPERIMENTS})
for comp_name in used_compressions:
    features = COMPRESSIONS[comp_name]["features"]
    n_channels = len(features) + 3
    assert n_channels % 2 == 0, f"Compression '{comp_name}' produces odd total channel count {n_channels}"
    compressed_cache_train[comp_name] = build_compressed_tensor(X_train_raw, features)
    compressed_cache_tune[comp_name] = build_compressed_tensor(X_tune_raw, features)
    compressed_cache_test[comp_name] = build_compressed_tensor(X_test_raw, features)
    print(f"[{comp_name}] features={features} -> "
          f"train {compressed_cache_train[comp_name].shape}, "
          f"tune {compressed_cache_tune[comp_name].shape}, "
          f"test {compressed_cache_test[comp_name].shape}")


[stat7] features=['mean', 'std', 'min', 'max', 'rms', 'skew', 'kurtosis'] -> train (3359, 60, 10), tune (593, 60, 10), test (988, 60, 10)


## Step 6 — Model builder (single Conv1D stack, no RNN)

**Layers unchanged from the previous version**, including the final `Dense(1, sigmoid)` output — only `Conv1D` /
`MaxPooling1D` / `Dense` (EFR32xG26 MVP accelerator constraint). Architecture is generalized to whatever channel
count the active compression config produces, and is defined entirely by `model_variants.<n>` in `config.json`.

**What did change:** the compile step now uses `mse` loss and `mae` metric instead of `binary_crossentropy`/`accuracy`,
to match the soft ordinal regression target (`config.json -> label.training_target`). The sigmoid output still
keeps every value in `[0, 1]`, which is exactly the range the regression targets `[0.0, 0.7, 1.0]` live in.


In [9]:
def build_model(window_frames, n_channels, model_cfg):
    layers = [Input(shape=(window_frames, n_channels))]
    for filters, kernel, stride, do_pool in zip(
        model_cfg["conv_filters"], model_cfg["kernel_sizes"], model_cfg["strides"], model_cfg["pool_after_layer"]
    ):
        layers.append(Conv1D(filters, kernel_size=kernel, strides=stride, padding="same", activation="relu"))
        if do_pool:
            layers.append(MaxPooling1D(2))
    layers.append(GlobalAveragePooling1D())
    for units in model_cfg["dense_units"]:
        layers.append(Dense(units, activation="relu"))
    layers.append(Dropout(model_cfg["dropout"]))
    layers.append(Dense(1, activation="sigmoid"))  # single output: severity score, regressed toward CLASS_TARGETS[label]
    model = Sequential(layers)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model


# Preview using the first experiment's compression/model combo
_preview_comp = COMPRESSIONS[EXPERIMENTS[0]["compression"]]["features"]
_preview_channels = len(_preview_comp) + 3
_preview_model_cfg = MODEL_VARIANTS[EXPERIMENTS[0]["model_variant"]]
build_model(WINDOW_FRAMES, _preview_channels, _preview_model_cfg).summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 60, 16)            816       
                                                                 
 max_pooling1d (MaxPooling1  (None, 30, 16)            0         
 D)                                                              
                                                                 
 conv1d_1 (Conv1D)           (None, 30, 32)            2592      
                                                                 
 max_pooling1d_1 (MaxPoolin  (None, 15, 32)            0         
 g1D)                                                            
                                                                 
 conv1d_2 (Conv1D)           (None, 15, 32)            3104      
                                                                 
 global_average_pooling1d (  (None, 32)                0

## Step 6b — Ordinal target encoding & threshold-tuning module (NEW)

Two small, self-contained pieces:

- `ordinal_regression_target(y)` — maps the true 3-class label to its fixed regression target
  (`CLASS_TARGETS[label]`, i.e. `0.0` / `0.7` / `1.0`). This is the *only* thing the model is fit against; it
  never sees `tau_1`/`tau_2` and the loss has no notion of "3 classes", just 3 numeric targets to regress toward.
- `scores_to_labels(scores, tau_1, tau_2)` — turns a continuous severity score into a 3-class prediction.
- `tune_thresholds(y_true, scores, ...)` — grid-searches `tau_1 <= tau_2` on the tuning split to maximize the
  configured metric (`config.json -> threshold_tuning.metric`, default macro-F1 across the 3 classes). Candidate
  thresholds are taken from `n_candidates` quantiles of the observed tuning scores (not a blind `linspace(0,1)`),
  so the search concentrates where the score distribution actually has mass.


In [10]:
def ordinal_regression_target(y):
    """y: (N,) int array with values in {0,1,2}. Returns (N,) float32 array with each entry replaced by
    CLASS_TARGETS[label] (e.g. 0.0/0.7/1.0) -- used as the MSE regression training target. Fancy-indexes
    CLASS_TARGETS directly so this stays correct if class_targets is ever reconfigured in config.json."""
    y = np.asarray(y)
    return CLASS_TARGETS[y].astype(np.float32)


def scores_to_labels(scores, tau_1, tau_2):
    """scores: (N,) float array in [0,1]. Returns (N,) int array in {0,1,2}.
    score < tau_1 -> 0 (Normal); tau_1 <= score < tau_2 -> 1 (Abnormal); score >= tau_2 -> 2 (Emergency).
    Must stay behaviourally identical to whatever comparison the deployment C code performs on the
    dequantized on-device score."""
    scores = np.asarray(scores)
    labels = np.zeros_like(scores, dtype=np.int64)
    labels[scores >= tau_1] = 1
    labels[scores >= tau_2] = 2
    return labels


def _score_metric(y_true, y_pred, metric):
    if metric == "f1_macro":
        return f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0)
    if metric == "accuracy":
        return accuracy_score(y_true, y_pred)
    raise ValueError(f"Unsupported threshold_tuning.metric '{metric}'")


def tune_thresholds(y_true, scores, metric=None, n_candidates=None):
    """Grid-search tau_1 <= tau_2 in [0,1] to maximize `metric` (default: config threshold_tuning.metric)
    on (y_true, scores) -- intended to be called on the TUNE split only.
    Returns (tau_1, tau_2, best_score, all_results) where all_results is a list of
    (tau_1, tau_2, score) tuples for every candidate pair evaluated (useful for plotting/debugging)."""
    metric = metric or TUNING_METRIC
    n_candidates = n_candidates or TUNING_N_CANDIDATES

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    quantiles = np.linspace(0.0, 1.0, n_candidates)
    candidates = np.unique(np.quantile(scores, quantiles))
    # Always allow the degenerate "never fires" / "always fires" ends of the range.
    candidates = np.unique(np.concatenate([[0.0], candidates, [1.0]]))

    best = (candidates[0], candidates[0], -np.inf)
    all_results = []
    for tau_1 in candidates:
        for tau_2 in candidates:
            if tau_2 < tau_1:
                continue
            y_pred = scores_to_labels(scores, tau_1, tau_2)
            s = _score_metric(y_true, y_pred, metric)
            all_results.append((float(tau_1), float(tau_2), float(s)))
            if s > best[2]:
                best = (tau_1, tau_2, s)

    tau_1, tau_2, best_score = best
    return float(tau_1), float(tau_2), float(best_score), all_results


## Step 7 — Export helpers: TFLite `.h`/`.cc`, and the compression module `.c`/`.h`

In [11]:
def export_cc_header(tflite_bytes, out_dir, model_name):
    array_name = f"{model_name}_int8_tflite"
    len_name = f"{array_name}_len"
    header_guard = f"{model_name.upper()}_H_"
    with open(os.path.join(out_dir, f"{model_name}.h"), "w") as f:
        f.write(f"#ifndef {header_guard}\n#define {header_guard}\n\n")
        f.write(f"extern const unsigned char {array_name}[];\n")
        f.write(f"extern const unsigned int {len_name};\n\n")
        f.write(f"#endif // {header_guard}\n")
    hex_array = ", ".join(f"0x{b:02x}" for b in tflite_bytes)
    with open(os.path.join(out_dir, f"{model_name}.cc"), "w") as f:
        f.write(f'#include "{model_name}.h"\n\n')
        f.write(f"const unsigned char {array_name}[] = {{\n{hex_array}\n}};\n\n")
        f.write(f"const unsigned int {len_name} = {len(tflite_bytes)};\n")


def export_compression_module(comp_name, features, out_dir="."):
    """Writes ppg_compress_<comp_name>.c/.h implementing the same statistical
    compression as compress_frame_stats()/build_compressed_tensor() above, for
    on-device use ahead of the TFLite model. Written to `out_dir` (default: notebook root).
    UNCHANGED: compression has no dependency on labels/classes."""
    module_name = f"ppg_compress_{comp_name}"
    header, source = generate_compression_c(
        features, module_name,
        window_frames=WINDOW_FRAMES, frame_samples=FRAME_SAMPLES, n_passthrough=3,
    )
    h_path = os.path.join(out_dir, f"{module_name}.h")
    c_path = os.path.join(out_dir, f"{module_name}.c")
    with open(h_path, "w") as f:
        f.write(header)
    with open(c_path, "w") as f:
        f.write(source)
    return h_path, c_path


## Step 8 — `run_experiment`: normalize, train (binary target), tune tau_1/tau_2, evaluate 3-class, quantize

Flow per experiment:
1. Normalize using **fit**-split statistics (unchanged).
2. Train the unchanged architecture against the soft ordinal regression target (`ordinal_regression_target`),
   with `mse` loss.
3. Predict raw sigmoid scores on the **tune** split; run `tune_thresholds` to pick `tau_1`/`tau_2`.
4. Predict raw sigmoid scores on the **test** split; apply the tuned thresholds; report full 3-class metrics
   (per-class precision/recall/F1, macro-F1, accuracy, 3x3 confusion matrix) — this is the only place test-set
   labels are used.
5. Quantize to int8 TFLite exactly as before (representative dataset drawn from the fit split).


In [12]:
def run_experiment(exp_cfg):
    name = exp_cfg["name"]
    comp_name = exp_cfg["compression"]
    features = COMPRESSIONS[comp_name]["features"]
    n_channels = len(features) + 3
    model_cfg = MODEL_VARIANTS[exp_cfg["model_variant"]]

    print(f"\n{'='*20} EXPERIMENT: {name} (compression={comp_name}, model={exp_cfg['model_variant']}) {'='*20}")

    X_train_c = compressed_cache_train[comp_name]
    X_tune_c = compressed_cache_tune[comp_name]
    X_test_c = compressed_cache_test[comp_name]

    mean = X_train_c.reshape(-1, n_channels).mean(axis=0)
    std = X_train_c.reshape(-1, n_channels).std(axis=0)
    std[std == 0] = 1.0
    X_train_n = (X_train_c - mean) / std
    X_tune_n = (X_tune_c - mean) / std
    X_test_n = (X_test_c - mean) / std

    y_train_target = ordinal_regression_target(y_train)

    model = build_model(WINDOW_FRAMES, n_channels, model_cfg)
    model.fit(
        X_train_n, y_train_target,
        epochs=TRAIN_CFG["epochs"], batch_size=TRAIN_CFG["batch_size"],
        validation_split=TRAIN_CFG.get("validation_split", 0.0), verbose=0,
    )

    # --- Threshold tuning on the held-out TUNE split (true 3-class labels, never touched by training) ---
    tune_scores = model.predict(X_tune_n, verbose=0).ravel()
    tau_1, tau_2, tuning_score, _ = tune_thresholds(y_tune, tune_scores)
    print(f"Tuned thresholds: tau_1={tau_1:.4f}, tau_2={tau_2:.4f} "
          f"({TUNING_METRIC} on tune split = {tuning_score:.4f})")

    # --- Final 3-class evaluation on the untouched TEST split ---
    test_scores = model.predict(X_test_n, verbose=0).ravel()
    y_pred_3class = scores_to_labels(test_scores, tau_1, tau_2)

    cm = confusion_matrix(y_test, y_pred_3class, labels=[0, 1, 2]).tolist()
    per_class_p, per_class_r, per_class_f1, per_class_support = precision_recall_fscore_support(
        y_test, y_pred_3class, labels=[0, 1, 2], zero_division=0
    )
    metrics = {
        "accuracy": float(accuracy_score(y_test, y_pred_3class)),
        "f1_macro": float(f1_score(y_test, y_pred_3class, labels=[0, 1, 2], average="macro", zero_division=0)),
        "per_class": {
            CLASS_NAMES[i]: {
                "precision": float(per_class_p[i]),
                "recall": float(per_class_r[i]),
                "f1": float(per_class_f1[i]),
                "support": int(per_class_support[i]),
            }
            for i in range(3)
        },
        "confusion_matrix": cm,
        "confusion_matrix_labels": CLASS_NAMES,
        "tau_1": tau_1,
        "tau_2": tau_2,
        "tuning_metric": TUNING_METRIC,
        "tuning_metric_value_on_tune_split": tuning_score,
        "n_train_windows": int(len(X_train_c)),
        "n_tune_windows": int(len(X_tune_c)),
        "n_test_windows": int(len(X_test_c)),
        "n_channels": int(n_channels),
    }
    print("Metrics:", json.dumps(metrics, indent=2))

    saved_model_dir = f"/tmp/{name}_saved_model"
    if os.path.exists(saved_model_dir):
        shutil.rmtree(saved_model_dir)
    model.save(saved_model_dir)

    def representative_dataset():
        for i in range(min(100, len(X_train_n))):
            yield [np.expand_dims(X_train_n[i], axis=0).astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_bytes = converter.convert()
    shutil.rmtree(saved_model_dir)

    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    in_d = interpreter.get_input_details()[0]
    out_d = interpreter.get_output_details()[0]

    return {
        "name": name, "config": exp_cfg, "metrics": metrics,
        "tflite_bytes": tflite_bytes, "tflite_size": len(tflite_bytes),
        "float_params": int(model.count_params()),
        "compression": comp_name, "features": features, "n_channels": n_channels,
        "mean": mean.tolist(), "std": std.tolist(),
        "tau_1": tau_1, "tau_2": tau_2,
        "input_quantization": {"scale": float(in_d["quantization"][0]), "zero_point": int(in_d["quantization"][1])},
        "output_quantization": {"scale": float(out_d["quantization"][0]), "zero_point": int(out_d["quantization"][1])},
        "model": model, # Added!!!
    }


## Step 9 — Run the full experiment grid

In [13]:
all_results = [run_experiment(exp) for exp in EXPERIMENTS]



==================== EXPERIMENT: stat7_base (compression=stat7, model=base) ====================
Tuned thresholds: tau_1=0.4714, tau_2=0.8768 (f1_macro on tune split = 0.9108)
Metrics: {
  "accuracy": 0.9574898785425101,
  "f1_macro": 0.9138447329126528,
  "per_class": {
    "Normal": {
      "precision": 0.9979550102249489,
      "recall": 0.9878542510121457,
      "f1": 0.9928789420142421,
      "support": 494
    },
    "Abnormal": {
      "precision": 0.9304556354916067,
      "recall": 0.9675810473815462,
      "f1": 0.9486552567237164,
      "support": 401
    },
    "Emergency": {
      "precision": 0.8536585365853658,
      "recall": 0.7526881720430108,
      "f1": 0.8,
      "support": 93
    }
  },
  "confusion_matrix": [
    [
      488,
      6,
      0
    ],
    [
      1,
      388,
      12
    ],
    [
      0,
      23,
      70
    ]
  ],
  "confusion_matrix_labels": [
    "Normal",
    "Abnormal",
    "Emergency"
  ],
  "tau_1": 0.47135263936004446,
  "tau_2": 0.87

INFO:tensorflow:Assets written to: /tmp/stat7_base_saved_model/assets
W0000 00:00:1789384491.122054    1813 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1789384491.122150    1813 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-09-14 18:14:51.122705: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/stat7_base_saved_model
2026-09-14 18:14:51.126718: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-09-14 18:14:51.126761: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/stat7_base_saved_model
2026-09-14 18:14:51.163813: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-09-14 18:14:51.165685: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-09-14 18:14:51.257205: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle

## Step 10 — Export top-K experiments to `models/<experiment_name>/`, and each used compression module to the notebook root

Each exported experiment's `config.json` now includes a `thresholds` block (`tau_1`, `tau_2`, the metric they were
tuned for, and the class boundary semantics) in addition to the existing normalization/quantization info — this is
everything the deployment team needs to reproduce the 3-way decision on-device.


In [14]:
ranking_metric = EXPORT_CFG["ranking_metric"]
top_k = EXPORT_CFG["top_k"]
models_dir = EXPORT_CFG["models_dir"]

ranked = sorted(all_results, key=lambda r: r["metrics"][ranking_metric], reverse=True)
top_results = ranked[:top_k]

os.makedirs(models_dir, exist_ok=True)
exported = []
for r in top_results:
    out_dir = os.path.join(models_dir, r["name"])
    os.makedirs(out_dir, exist_ok=True)
    r["model"].save(os.path.join(out_dir, "model_float.h5")) ## Added!!!
    with open(os.path.join(out_dir, "model_int8.tflite"), "wb") as f:
        f.write(r["tflite_bytes"])
    export_cc_header(r["tflite_bytes"], out_dir, "model")

    full_config = {
        "data": DATA_CFG, "channels": CH_CFG, "label": LBL_CFG,
        "compression": {"name": r["compression"], "features": r["features"]},
        "model_variant": MODEL_VARIANTS[r["config"]["model_variant"]],
        "training": TRAIN_CFG,
        "normalization": {
            "mean": r["mean"], "std": r["std"],
            "channel_order": r["features"] + ["spo2", "bpm", "motion_level"],
        },
        "input_quantization": r["input_quantization"],
        "output_quantization": r["output_quantization"],
        "thresholds": {
            "tau_1": r["tau_1"],
            "tau_2": r["tau_2"],
            "tuned_for_metric": THRESH_CFG["metric"],
            "score_domain": "float sigmoid output in [0, 1], AFTER dequantizing the int8 model output "
                             "with output_quantization.scale/zero_point",
            "decision_rule": "score < tau_1 -> Normal (0); tau_1 <= score < tau_2 -> Abnormal (1); "
                              "score >= tau_2 -> Emergency (2)",
            "class_names": CLASS_NAMES,
        },
    }
    with open(os.path.join(out_dir, "config.json"), "w") as f:
        json.dump(full_config, f, indent=2)
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(r["metrics"], f, indent=2)

    exported.append(r["name"])
    print(f"Exported '{r['name']}' -> {out_dir}/  ({r['tflite_size']/1024:.2f} KB, "
          f"{r['float_params']} params, {r['n_channels']} channels, "
          f"tau_1={r['tau_1']:.3f}, tau_2={r['tau_2']:.3f}, "
          f"{ranking_metric}={r['metrics'][ranking_metric]:.3f})")

# Export the compression C module for every compression config actually used (not just top-K models,
# since the same compression config may back multiple experiments/models). UNCHANGED step.
print()
for comp_name in used_compressions:
    features = COMPRESSIONS[comp_name]["features"]
    h_path, c_path = export_compression_module(comp_name, features, out_dir=".")
    print(f"Exported compression module '{comp_name}' -> {h_path}, {c_path}")


Exported 'stat7_base' -> models/stat7_base/  (16.64 KB, 7601 params, 10 channels, tau_1=0.471, tau_2=0.877, f1_macro=0.914)

Exported compression module 'stat7' -> ./ppg_compress_stat7.h, ./ppg_compress_stat7.c


/home/learncoding0111/anaconda3/envs/iot-env/lib/python3.10/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


## Step 11 — Final comparison table (all experiments)

In [15]:
comparison_rows = []
for r in ranked:
    row = {
        "name": r["name"], "compression": r["compression"],
        "model_variant": r["config"]["model_variant"], "n_channels": r["n_channels"],
        "accuracy": r["metrics"]["accuracy"],
        "f1_macro": r["metrics"]["f1_macro"],
        "f1_normal": r["metrics"]["per_class"]["Normal"]["f1"],
        "f1_abnormal": r["metrics"]["per_class"]["Abnormal"]["f1"],
        "f1_emergency": r["metrics"]["per_class"]["Emergency"]["f1"],
        "recall_emergency": r["metrics"]["per_class"]["Emergency"]["recall"],
        "tau_1": r["tau_1"], "tau_2": r["tau_2"],
        "n_train_windows": r["metrics"]["n_train_windows"],
        "n_tune_windows": r["metrics"]["n_tune_windows"],
        "n_test_windows": r["metrics"]["n_test_windows"],
    }
    row["tflite_kb"] = round(r["tflite_size"] / 1024, 2)
    row["params"] = r["float_params"]
    row["exported"] = r["name"] in exported
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
cols_order = ["name", "compression", "model_variant", "n_channels",
              "accuracy", "f1_macro", "f1_normal", "f1_abnormal", "f1_emergency", "recall_emergency",
              "tau_1", "tau_2",
              "n_train_windows", "n_tune_windows", "n_test_windows", "params", "tflite_kb", "exported"]
comparison_df = comparison_df[[c for c in cols_order if c in comparison_df.columns]]

pd.set_option("display.width", 200)
print("=" * 100)
print(f"COMPARISON — ranked by '{ranking_metric}' (descending)")
print("=" * 100)
print(comparison_df.to_string(index=False))

comparison_df.to_csv("experiment_comparison.csv", index=False)
print("\nSaved full comparison table to experiment_comparison.csv")
print(f"\nTrain(fit) windows: {len(X_train_raw)} | Tune windows: {len(X_tune_raw)} | Test windows: {len(X_test_raw)} "
      f"(random stratified splits, seed={DATA_CFG['seed']}/{TUNING_SEED}, "
      f"test_fraction={DATA_CFG['test_fraction']}, tuning_fraction={TUNING_FRACTION})")


COMPARISON — ranked by 'f1_macro' (descending)
      name compression model_variant  n_channels  accuracy  f1_macro  f1_normal  f1_abnormal  f1_emergency  recall_emergency    tau_1   tau_2  n_train_windows  n_tune_windows  n_test_windows  params  tflite_kb  exported
stat7_base       stat7          base          10   0.95749  0.913845   0.992879     0.948655           0.8          0.752688 0.471353 0.87682             3359             593             988    7601      16.64      True

Saved full comparison table to experiment_comparison.csv

Train(fit) windows: 3359 | Tune windows: 593 | Test windows: 988 (random stratified splits, seed=42/42, test_fraction=0.2, tuning_fraction=0.15)
